# Düğüm Özellik Normalizasyon Düzeltmesi

**Sorun:** `nodes.parquet`'te ham (normalize edilmemiş) sütunlar var:
- `tpop`, `e_pop` → milyonlar
- `milex` → milyonlar
- `milper` → binler
- `e_gdppc` → binler-yüz binler

`log_*` versiyonları zaten var. Bu notebook ham sütunları drop edip snapshot'ları yeniden üretir.

**Süre:** ~2-3 dakika.

**Yan etki:** `data/snapshots/graph_*.pt` dosyaları üzerine yazılır. Eski model checkpoint'leri bu yüzden geçersiz hale gelir; 04a ve 04'ü tekrar çalıştırman gerekir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')

START_YEAR, END_YEAR = 1946, 2016

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
from tqdm.auto import tqdm

In [ ]:
# 1. nodes.parquet'i oku, mevcut sütunları gör
nodes = pd.read_parquet(os.path.join(CANON_DIR, 'nodes.parquet'))
print(f'Toplam sütun sayısı: {len(nodes.columns)}')
print(f'\nSütunlar:')
for c in nodes.columns:
    if c in ['year', 'cow_code', 'state_name']:
        continue
    if nodes[c].dtype in [np.float32, np.float64, np.int64, np.int32]:
        col = nodes[c].dropna()
        if len(col) > 0:
            print(f'  {c:20s} → min={col.min():.2e}, max={col.max():.2e}, mean={col.mean():.2e}')

In [ ]:
# 2. Ham (normalize edilmemiş) sütunları drop et
# log_* versiyonları zaten var, onlar yeterli
DROP_COLS = ['tpop', 'milex', 'milper', 'e_gdppc', 'e_pop',
             'irst', 'pec', 'upop']  # ham büyük değerler

to_drop = [c for c in DROP_COLS if c in nodes.columns]
print(f'Drop edilecek sütunlar: {to_drop}')
nodes_clean = nodes.drop(columns=to_drop)

# Hâlâ ölçek farkı olan sütunları z-skor normalize et
# polity2 zaten [-10, 10], v2x_* zaten [0,1], log_* zaten log ölçeğinde — bunları dokunma
# Ama cinc gibi [0,1] aralığındaki sütunlar bile farklı dağılımda, hepsini standartlaştır

numeric_feature_cols = [c for c in nodes_clean.columns
                        if c not in ['year', 'cow_code', 'state_name']
                        and nodes_clean[c].dtype in [np.float32, np.float64, np.int64, np.int32]]

print(f'\nKalan sayısal özellik sütunları: {len(numeric_feature_cols)}')
print(numeric_feature_cols)

In [ ]:
# 3. z-skor normalizasyon: tüm panel üzerinde, sütun başına
# Eğitim verisinden hesaplanan ortalama/std kullanıp tüm panele uygula
# (zamansal sızıntıyı önlemek için sadece train döneminden hesaplanır)
TRAIN_END = 1999
train_mask = nodes_clean['year'] <= TRAIN_END

stats = {}
for col in numeric_feature_cols:
    train_vals = nodes_clean.loc[train_mask, col].dropna()
    mean = train_vals.mean()
    std = train_vals.std()
    if std < 1e-6 or pd.isna(std):
        std = 1.0
    stats[col] = (mean, std)
    nodes_clean[col] = (nodes_clean[col] - mean) / std

print('Z-skor normalizasyon sonrası özet:')
for col in numeric_feature_cols:
    vals = nodes_clean[col].dropna()
    print(f'  {col:20s} → min={vals.min():.2f}, max={vals.max():.2f}, mean={vals.mean():.3f}, std={vals.std():.2f}')

# Güncellenmiş nodes.parquet'i kaydet
nodes_clean.to_parquet(os.path.join(CANON_DIR, 'nodes.parquet'))
print(f'\n✓ nodes.parquet güncellendi ({len(nodes_clean.columns)} sütun)')

In [ ]:
# 4. Kenar tablolarını yükle (değişmiyorlar)
edge_tables = {}
for rel, fname in [('allies', 'edges_ally.parquet'),
                   ('trades', 'edges_trade.parquet'),
                   ('votes_with', 'edges_vote.parquet'),
                   ('conflicts_with', 'edges_conflict.parquet')]:
    fpath = os.path.join(CANON_DIR, fname)
    if os.path.exists(fpath):
        edge_tables[rel] = pd.read_parquet(fpath)

feature_cols = [c for c in nodes_clean.columns
                if c not in ['year', 'cow_code', 'state_name']
                and nodes_clean[c].dtype in [np.float32, np.float64, np.int64, np.int32]]
print(f'Snapshot üretiminde kullanılacak özellik sütunları: {len(feature_cols)}')
print(feature_cols)

In [ ]:
# 5. Snapshot'ları yeniden inşa et (02_preprocessing'deki build_snapshot ile aynı mantık)
def build_snapshot(year):
    nodes_y = nodes_clean[nodes_clean['year'] == year].copy()
    if len(nodes_y) == 0:
        return None
    
    nodes_y = nodes_y.sort_values('cow_code').reset_index(drop=True)
    cow_to_idx = {int(c): i for i, c in enumerate(nodes_y['cow_code'])}
    
    x_df = nodes_y[feature_cols].copy()
    x_mask = (~x_df.isna()).astype(float).values
    x_filled = x_df.fillna(0.0).values  # z-skor ortalaması 0
    
    data = HeteroData()
    data['country'].x = torch.tensor(x_filled, dtype=torch.float)
    data['country'].x_mask = torch.tensor(x_mask, dtype=torch.float)
    data['country'].cow_code = torch.tensor(nodes_y['cow_code'].values, dtype=torch.long)
    
    for rel, etable in edge_tables.items():
        etable_y = etable[etable['year'] == year]
        etable_y = etable_y[
            etable_y['cow_i'].isin(cow_to_idx) & etable_y['cow_j'].isin(cow_to_idx)
        ]
        if len(etable_y) == 0:
            continue
        
        src = etable_y['cow_i'].map(cow_to_idx).values
        dst = etable_y['cow_j'].map(cow_to_idx).values
        edge_index = torch.tensor(
            np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])]),
            dtype=torch.long
        )
        edge_weight = torch.tensor(
            np.concatenate([etable_y['weight'].values, etable_y['weight'].values]),
            dtype=torch.float
        )
        data['country', rel, 'country'].edge_index = edge_index
        data['country', rel, 'country'].edge_attr = edge_weight.unsqueeze(1)
    
    data.year = year
    return data

n_built = 0
for year in tqdm(range(START_YEAR, END_YEAR + 1), desc='Snapshot yeniden inşa'):
    snap = build_snapshot(year)
    if snap is None:
        continue
    torch.save(snap, os.path.join(SNAP_DIR, f'graph_{year}.pt'))
    n_built += 1

print(f'\n✓ {n_built} snapshot yeniden inşa edildi')

In [ ]:
# 6. Sanity check — yeni snapshot'ın özellik aralığı
sample = torch.load(os.path.join(SNAP_DIR, 'graph_1990.pt'), weights_only=False)
x = sample['country'].x
print(f'1990 düğüm özellik tensörü:')
print(f'  Shape: {x.shape}')
print(f'  Genel: min={x.min().item():.2f}, max={x.max().item():.2f}, mean={x.mean().item():.3f}, std={x.std().item():.2f}')
print(f'\nHer sütun için:')
for i in range(x.shape[1]):
    col = x[:, i]
    print(f'  Sütun {i:2d} ({feature_cols[i]:15s}): min={col.min().item():+.2f}, max={col.max().item():+.2f}, mean={col.mean().item():+.3f}, std={col.std().item():.2f}')

## Sonraki adım

Tüm sütunlar artık standart normal aralıkta olmalı (yaklaşık -3 ile +3 arası, ortalama 0, std 1).

**Şimdi sırasıyla:**
1. `04a_pretrain.ipynb`'i baştan çalıştır → feat_loss artık trilyon değil makul (~1-2 başlangıç, ~0.3 son) olmalı
2. `04_model.ipynb`'i baştan çalıştır → encoder'ın temiz girdiler üzerinde gerçek bir başlangıç olur